In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Employees


In [0]:
df_emp = spark.read.table('project_cata.bronze.employees')
# df_emp.printSchema()
# df_nulls = df_emp.select([sum(when(col(i).isNull() | (trim(col(i)) == ""), 1).otherwise(0)).alias(i) for i in df_emp.columns])
# display(df_nulls)

# converting date to valid date formats
formats = ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy"]
df_emp = df_emp.withColumn('date_of_birth', coalesce(*(try_to_date(col('date_of_birth'), f) for f in formats))).withColumn('hire_date', coalesce(*(try_to_date(col('hire_date'), f) for f in formats))).withColumn('last_updated', to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss"))

#removing duplicates
df_emp = df_emp.withColumn('rono', row_number().over(Window.partitionBy('employee_id').orderBy(col('last_updated').desc())))
df_emp = df_emp.filter(col('rono') == 1).drop('rono')
# df_dups = df_emp.groupBy('employee_id').agg(count('employee_id').alias('ecount')).filter(col('ecount')>1)
# display(df_dups)

#Replacing invalid emails with Nulls
eregex = r'^[A-Za-z0-9_.-]+@[A-Za-z0-9_.-]+\.[A-Za-z0-9_]{2,}$'
df_emp = df_emp.withColumn('email', when(col('email').rlike(eregex), col('email')).otherwise(None))
# display(df_emp.filter(col('email').isNull()))

#checking any employees without department
df_dep = spark.read.table('project_cata.bronze.departments')
df_dep = df_dep.withColumn('sub_departments', explode(col('sub_departments.department'))).withColumn('sub_dep_id', col('sub_departments._id')).withColumn('sub_dep_name', col('sub_departments.name'))

df_emp = df_emp.join(df_dep.select('sub_dep_id'), df_emp.department_id == df_dep.sub_dep_id, 'left').withColumn('reject_reason', when(col('sub_dep_id').isNull(), 'No Dep Id').when(col('hire_date').isNull(), 'No hire date').otherwise(None)).drop('sub_dep_id')

#Converting salry to float
df_emp = df_emp.withColumn('base_salary', round( regexp_replace(col('base_salary'), ',' , '').try_cast('float'), 2))

quatantine = df_emp.filter(col('reject_reason').isNotNull())
df_emp = df_emp.filter(col('reject_reason').isNull())

# Insert data to silver Employees
from delta.tables import DeltaTable

if not spark.catalog.tableExists('project_cata.silver.employees'):
    df_emp.write.format('delta').saveAsTable('project_cata.silver.employees')
else:
    target = DeltaTable.forName(spark , 'project_cata.silver.employees')
    target.alias("t").merge(df_emp.alias("s"), 't.employee_id=s.employee_id')\
        .whenMatchedUpdateAll(condition='s.last_updated >= t.last_updated ')\
            .whenNotMatchedInsertAll()\
                .execute()

## Departments

In [0]:
df_dep = spark.read.table('project_cata.bronze.departments')

#Flattern the JSON
df_parent_dep = df_dep.select(col('_id').alias('department_id'), col('name').alias('department_name'), col('_cost_center').alias('cost_center'), lit(None).alias('parent_department_id'))
# display(df_dep)

df_child_dep = df_dep.select(col('_id').alias('parent_department_id'), explode(col('sub_departments.department')).alias('d'))\
    .select(col('parent_department_id'), col('d._cost_center').alias('cost_center'), col('d._id').alias('department_id'), col('d.name').alias('department_name'))

df_dep = df_parent_dep.unionByName(df_child_dep)

df_dep = df_dep.filter((col('department_id').isNotNull()) & (col('parent_department_id').isNull() | (col('department_id') != col('parent_department_id'))) & (col('cost_center').isNotNull()))

df_dep.write.format('delta').mode('overwrite').saveAsTable('project_cata.silver.departments')

## Job Roles

In [0]:
df_job_roles = spark.read.table('project_cata.bronze.job_roles')

df_job_roles = df_job_roles.select(col('_id').alias('job_role_id'), col('title').alias('job_title'), col('job_family'), col('_grade').alias('grade'), col('_hourly').alias('is_hourly'), col('department_id'), col('people_manager').alias('is_people_manager'), col('salary_band.min').alias('salary_min'), col('salary_band.max').alias('salary_max'), col('salary_band._currency').alias('salary_currency'))

dupes = df_job_roles.groupBy('job_role_id').agg(count('job_role_id').alias('count')).filter(col('count')>1)

if dupes.count() > 1:
    df_job_roles.withColumn('ccount', rank('job_role_id').over(Window.partitionBy('job_role_id'))).filter(col('ccount')==1).drop('ccount')

df_job_roles = df_job_roles.withColumn('is_hourly', when(((col('is_hourly') == 'Y') | (col('is_hourly') == 'true')) , True).when(((col('is_hourly') == 'N') | (col('is_hourly') == 'false')) , False)).filter((col('salary_min') > 0) & (col('salary_max') > col('salary_min')))

df_dep = spark.read.table('project_cata.silver.departments')
df_job_roles = df_job_roles.join(df_dep , on=['department_id'] , how='left').filter(col('department_name').isNotNull())\
    .drop('department_name', 'cost_center', 'parent_department_id')

df_job_roles.write.format('delta').mode('overwrite').saveAsTable('project_cata.silver.job_roles')


## Locations

In [0]:
df_locations = spark.read.table('project_cata.bronze.locations')

df_locations = df_locations.filter(col('name').isNotNull()).select(col('_id').alias('location_id'), col('name').alias('location_name'), col('_status').alias('status'), col('address.city').alias('city'), col('address.country_code').alias('country_code'), col('address.region').alias('region'), col('facility._type').alias('facility_type'), col('facility.headcount_capacity').alias('headcount_capacity'), col('facility.timezone').alias('timezone'), col('is_manufacturing_site'), col('local_currency')).distinct()

df_locations.write.format('delta').mode('overwrite').saveAsTable('project_cata.silver.locations')


## Employee Changes

In [0]:
df_emp_changes = spark.read.table('project_cata.bronze.employee_changes')
df_emp_changes.printSchema()
display(df_emp_changes)